In [3]:
import numpy as np
from scipy.optimize import minimize
import sympy as sp

In [81]:
import numpy as np

# Use a single complex dtype for numpy everywhere.
DTYPE = np.complex128

INV_SQRT2 = 1.0 / np.sqrt(2.0)
H = INV_SQRT2 * np.array([[1, 1], [1, -1]], dtype=DTYPE)

# LAMBDA_PI is the base rotation angle realized by the H/T building blocks:
# cos(LAMBDA_PI) = cos^2(pi/8) = (1 + 1/sqrt2)/2. Because LAMBDA_PI / (2 pi) is
# irrational, the multiples {k * LAMBDA_PI mod 2 pi} densely fill [0, 2 pi).
LAMBDA_PI = np.arccos((1.0 + INV_SQRT2) / 2.0)
TWO_PI = 2.0 * np.pi


class Bloch:
    """Axis-angle (Bloch) form of a 2x2 unitary G:

        G = e^{i alpha} (cos(theta/2) I - i sin(theta/2) (n . sigma))

    i.e. a global phase e^{i alpha} times a rotation by angle `theta` about the
    Bloch-sphere axis `n`. Here (n . sigma) = n_x X + n_y Y + n_z Z.
    """

    alpha: float  # global phase
    n: np.ndarray  # unit rotation axis, shape (3,): [n_x, n_y, n_z]
    theta: float  # rotation angle

    def __init__(self,alpha,n,theta):
        self.alpha =  np.real_if_close(alpha)
        self.n = np.real_if_close(n)
        self.theta = np.real_if_close(theta)
    def display(self):
        print("alpha(phase Angle) is {},\n the unit vector is nx={}  ny={}  nz={},\n the angle of rotation is{}".format(self.alpha,self.n[0],self.n[1],self.n[2],self.theta))

In [5]:
def to_bloch(g: np.ndarray) -> Bloch:
    """Recover the Bloch form (alpha, n, theta) of a 2x2 unitary `g`."""
    x = np.linalg.det(g)
    alpha = np.arctan2(x.imag,x.real)*(1/2)
    g_thilda = np.exp(-1j * alpha)*g
    theta = 2*np.arccos(np.trace(g_thilda)/2)
    n =[]
    sigma = [np.array([[0,1],[1,0]]),np.array([[0,-1j],[1j,0]]),np.array([[1,0],[0,-1]])]
    for sigmas in sigma:
        matrix = sigmas @ g_thilda
        trace = (np.trace(matrix)*(1j))/2
        nj = trace/np.sin(theta/2)
        n.append(nj)
    n_array = np.array(n)
    ans = Bloch(alpha,n_array,theta)
    return ans
Bloch_ = to_bloch(H)
Bloch_.display()

alpha(phase Angle) is 1.5707963267948966,
 the unit vector is nx=0.7071067811865475  ny=0.0  nz=0.7071067811865475,
 the angle of rotation is3.141592653589793


In [6]:
a = np.linalg.norm(Bloch_.n)
type(a)

numpy.float64

In [7]:
cot = np.cos(np.pi/8.0)/np.sin(np.pi/8)
n1 = np.array([-cot/np.sqrt(1+2*cot*cot),1/np.sqrt(1+2*cot*cot),cot/np.sqrt(1+2*cot*cot)])
n2 = np.array([1/(np.sqrt(2)*np.sqrt(1+2*cot*cot)),(np.sqrt(2)*cot)/(np.sqrt(1+2*cot*cot)),
          (-1*1)/(np.sqrt(2)*np.sqrt(1+2*cot*cot))])

# frame derived from the axes (given)
# take the dot product of the Bloch axis with these
# the minus sign arises from the double cover issue
n1 = -n1
n2 = -n2
n3 = np.cross(n1, n2)



# def Cost_function(angles: tuple[float,float,float],b:Bloch)->np.float64:
#     alpha,beta,gamma = angles
#     n = b.n
#     phi = b.theta
#     R1 = np.array([[np.cos(alpha/2)-1j*n1[2]*np.sin(alpha/2),-1j*n1[0]*np.sin(alpha/2)-n1[1]*np.sin(alpha/2)],
#         [n1[1]*np.sin(alpha/2)-1j*n1[0]*np.sin(alpha/2), np.cos(alpha/2)+1j*n1[2]*np.sin(alpha/2)]])
#     R2 = np.array([[np.cos(beta/2)-1j*n2[2]*np.sin(beta/2),-1j*n2[0]*np.sin(beta/2)-n2[1]*np.sin(beta/2)],
#         [n2[1]*np.sin(beta/2)-1j*n2[0]*np.sin(beta/2), np.cos(beta/2)+1j*n2[2]*np.sin(beta/2)]])
#     R3 = np.array([[np.cos(gamma/2)-1j*n1[2]*np.sin(gamma/2),-1j*n1[0]*np.sin(gamma/2)-n1[1]*np.sin(gamma/2)],
#         [n1[1]*np.sin(gamma/2)-1j*n1[0]*np.sin(gamma/2), np.cos(gamma/2)+1j*n1[2]*np.sin(gamma/2)]])
#     U_tilde = np.array([[np.cos(phi/2)-1j*n[2]*np.sin(phi/2),-1j*n[0]*np.sin(phi/2)-n[1]*np.sin(phi/2)],
#             [n[1]*np.sin(phi/2)-1j*n[0]*np.sin(phi/2), np.cos(phi/2)+1j*n[2]*np.sin(phi/2)]])
#     M_guess =R1@R2@R3 

#     error = np.linalg.norm(U_tilde - M_guess)

#     return error



def n1n2n1_angles(b: Bloch) -> tuple[float, float, float, float]:
    """Factor the rotation part of a unitary (given as its Bloch form `b`) as
        u = e^{i global_phase} * Rn1(alpha) * Rn2(beta) * Rn1(gamma)

    where Ra(angle) is a rotation by `angle` about axis a, and {a1, a2, a3} is
    the orthonormal frame defined above. Returns (alpha, beta, gamma, global_phase).
    """
    #TODO(student): implement using the steps above.
    # best_error = float('inf')
    # best_angles = None

    # # Drop 100 "hikers" at random coordinates
    # for i in range(100):
    #     # Generate 3 random angles between 0 and 2*pi
    #     random_guess = np.random.uniform(0, 2 * np.pi, 3)
    
    #     # Run the optimizer from this new random spot
    #     result = minimize(Cost_function, random_guess, args=(b,))
    
    #     # If this hiker found a deeper valley than the previous best, save it!
    #     if result.fun < best_error:
    #         best_error = result.fun
    #         best_angles = result.x
        
    #     # If we hit an error basically indistinguishable from zero, stop searching!
    #     if best_error< 1e-7:
    #         break

    # alpha = best_angles[0]
    # beta = best_angles[1]
    # gamma = best_angles[2]
    # print(best_error)
    """
    Solves the exact analytical equations for the n1-n2-n1 SU(2) decomposition.
    """
    # 1. THE FIX: Map to the half-angle space required by the equations
    phi_half = b.theta / 2.0
    phase = b.alpha
    n = np.array(b.n, dtype=float)
    
    # 2. Define the known vectors and scalars using phi_half!
    n3 = np.cross(n1, n2)
    V = n * np.sin(phi_half)
    D = np.cos(phi_half)
    
    # 3. Build the 3x3 basis matrix and solve for A, B, C
    basis_matrix = np.column_stack((n1, n2, n3))
    A, B, C = np.linalg.solve(basis_matrix, V)
    
    # 4. Extract the half-angle combinations
    # (Note: If A and D are 0, atan2 handles the singularity perfectly!)
    gamma_plus_alpha = np.arctan2(A, D)
    gamma_minus_alpha = np.arctan2(C, B)
    
    # 5. Extract beta_half using vector magnitudes
    cos_beta_half = np.linalg.norm([A, D])
    sin_beta_half = np.linalg.norm([B, C])
    beta_half = np.arctan2(sin_beta_half, cos_beta_half)
    
    # 6. Algebraically isolate alpha_half and gamma_half
    gamma_half = (gamma_plus_alpha + gamma_minus_alpha) / 2.0
    alpha_half = (gamma_plus_alpha - gamma_minus_alpha) / 2.0
    
    # 7. THE FIX: Scale the half-angles back up to the full hardware angles!
    alpha = alpha_half * 2.0
    beta = beta_half * 2.0
    gamma = gamma_half * 2.0
    
    # 8. Clean up angles to strictly sit in the [0, 2*pi] circle
    alpha, beta, gamma = np.mod([alpha, beta, gamma], 2 * np.pi)
    
    return (phase, alpha, beta, gamma)

In [8]:
n1n2n1_angles(Bloch_)

(array(1.57079633),
 np.float64(1.5707963267948966),
 np.float64(3.141592653589793),
 np.float64(4.71238898038469))

In [9]:
def approx_angle_with_tolerance(angle: float, tolerance: float) -> int:
    """Find an integer multiple k such that
        (k * LAMBDA_PI) mod 2*pi  ~=  angle   (within `tolerance`)
    Since LAMBDA_PI / (2 pi) is irrational, such a k always exists; search
    k = 1, 2, 3, ... and return the first one whose wrapped multiple lands within
    `tolerance` of `angle` (compare both as angles in [0, 2 pi)).

    Hint:
      * wrap an angle into [0, 2 pi)
      * the angular distance between two wrapped angles a, b is
        min(|a - b|, TWO_PI - |a - b|) (so 0.01 and 2*pi - 0.01 count as close).
    """
    # TODO(student): implement using the hint above.
    multiple_angle = LAMBDA_PI
    i=2
    while(min(abs(multiple_angle-angle),TWO_PI-abs(multiple_angle-angle))>tolerance):
        multiple_angle = i*multiple_angle
        multiple_angle = np.mod(multiple_angle, 2 * np.pi)
        i=i+1
    
    return i

In [10]:
approx_angle_with_tolerance(float(np.pi),0.000001)

6267648

In [11]:
def decompose_2x2(u: np.ndarray, tolerance: float) -> tuple[int, int, int]:
   
    u_bloch = to_bloch(u)
    phase, alpha, beta, gamma = n1n2n1_angles(u_bloch)
    print(alpha,beta,gamma)
    k= approx_angle_with_tolerance(alpha,tolerance)
    l= approx_angle_with_tolerance(beta,tolerance)
    m= approx_angle_with_tolerance(gamma,tolerance)

    return (k,l,m)

In [12]:
X = np.array([[0,1],[1,0]])
decompose_2x2(X,1e-6)

3.4156068573999496 1.649887332043646 6.00917110336943


(457569, 1780164, 1214949)

In [15]:
sigma = np.array([np.array([[0,1],[1,0]]),np.array([[0,-1j],[1j,0]]),np.array([[1,0],[0,-1]])])

axis = np.array([1,2,3])

(axis[0]*sigma[0]+axis[1]*sigma[1]+axis[2]*sigma[2])

array([[ 3.+0.j,  1.-2.j],
       [ 1.+2.j, -3.+0.j]])

In [36]:
def from_axis_angle(b: Bloch) -> np.ndarray:
    """Build a 2x2 unitary from its Bloch form: a global phase times a rotation
    by angle b.theta about axis b.n (inverse of to_bloch).

        G = e^{i b.alpha} (cos(b.theta/2) I - i sin(b.theta/2) (b.n . sigma))

    where (b.n . sigma) = n_x X + n_y Y + n_z Z. Assumes b.n is a unit vector.
    """
    # TODO: implement using the formula above.
    phase = b.alpha
    axis = b.n
    theta = b.theta
    I = np.array([[1,0],[0,1]])
    G_without_phase = (np.cos(theta/2)*I) -1j*np.sin(theta/2)*(axis[0]*sigma[0]+axis[1]*sigma[1]+axis[2]*sigma[2])
    G = np.exp(1j*phase)*G_without_phase
    G = np.round(G,decimals=10)
    return G

In [37]:
_bloch = to_bloch(np.array([[0,1],[1,0]]))
print(_bloch.theta)

3.141592653589793


In [38]:
G = from_axis_angle(_bloch)

In [39]:
print(G)

[[0.+0.j 1.-0.j]
 [1.-0.j 0.+0.j]]


In [75]:
def Rz(theta: float) -> np.ndarray:
    """Rotation about the z axis (no global phase):

    Rz(theta) = diag(e^{-i theta/2}, e^{i theta/2}).
    """
    # TODO: implement (hint: from_axis_angle about axis [0, 0, 1]).
    n = np.array([0,0,1])
    Rz_theta = (np.cos(theta/2)*I) -1j*np.sin(theta/2)*(n[0]*sigma[0]+n[1]*sigma[1]+n[2]*sigma[2])
    Rz_theta = np.round(Rz_theta,decimals=10)
    return Rz_theta



def Ry(theta: float) -> np.ndarray:
    """Rotation about the y axis (no global phase):

    Ry(theta) = [[cos(theta/2), -sin(theta/2)], [sin(theta/2), cos(theta/2)]].
    """
    # TODO: implement (hint: from_axis_angle about axis [0, 1, 0]).
    n = np.array([0,1,0])
    Ry_theta = (np.cos(theta/2)*I) -1j*np.sin(theta/2)*(n[0]*sigma[0]+n[1]*sigma[1]+n[2]*sigma[2])
    Ry_theta = np.round(Ry_theta,decimals=10)
    return Ry_theta


def euler_angles_zyz(u: np.ndarray) -> tuple[float, float, float, float]:
    """ZYZ Euler decomposition of a 2x2 unitary: angles (alpha, beta, gamma, delta)
    with

        u = e^{i alpha} Rz(beta) Ry(gamma) Rz(delta).

    alpha is the global phase (arg(det u)/2); the rest come from S = e^{-i alpha} u
    in SU(2), where s00 = cos(gamma/2) e^{-i(beta+delta)/2} and
    s10 = sin(gamma/2) e^{i(beta-delta)/2}. When gamma = 0 (s10 = 0), beta/delta are
    split arbitrarily (gimbal lock); the identity still holds.
    """
    # TODO: implement using the relations above.
    x = np.linalg.det(u)
    alpha = np.arctan2(x.imag, x.real)*(1/2)
    S = np.exp(-1j*alpha)*u
    gamma = 2*np.arccos(np.abs(S[0,0]))
    if np.isclose(np.sin(gamma/2), 0.0):
        Beta = 0.0
        Delta = np.real(-1j * 2 * np.log(S[1,1]))

    elif np.isclose(np.cos(gamma/2), 0.0):
        Beta = 0.0
        Delta = np.real(1j * 2 * np.log(S[1,0]))

    else:
        Beta = np.real(-1j * np.log( S[1,0] * S[1,1] / (np.sin(gamma/2) * np.cos(gamma/2)) ))
        Delta = np.real(-1j * np.log( -1 * (S[0,1] * S[1,1]) / (np.sin(gamma/2) * np.cos(gamma/2)) ))
    alpha = float(np.round(np.real(alpha), decimals=10))
    Beta = float(np.round(Beta, decimals=10))
    gamma = float(np.round(np.real(gamma), decimals=10))
    Delta = float(np.round(Delta, decimals=10))
    angles = tuple([alpha,Beta,gamma,Delta])
    return angles

In [76]:
Z = np.array([[0,1],[1,0]])
print(euler_angles_zyz(Z))

(1.5707963268, 0.0, 3.1415926536, 3.1415926536)


In [78]:
def unitary2_sqrt(u: np.ndarray) -> np.ndarray:
    """Principal square root: a 2x2 unitary V with V @ V == u, phase included.
    Take the Bloch form of u and halve both alpha and theta (same axis); squaring
    back doubles them, reproducing u exactly.
    """
    # TODO: implement (hint: to_bloch, halve alpha and theta, from_axis_angle).
    b = to_bloch(u)
    alpha_half = b.alpha/2
    theta_half = b.theta/2
    n = b.n 
    b_sqrt = Bloch(alpha_half,n,theta_half)
    b_sqrt_matrix = from_axis_angle(b_sqrt)
    return b_sqrt_matrix

In [82]:
Z = np.array([[0,1],[1,0]])
unitary2_sqrt(Z)

array([[0.5+0.5j, 0.5-0.5j],
       [0.5-0.5j, 0.5+0.5j]])